此版本加入使用SCIRTRAN构建的O3 H2O 气体吸收查找表

In [9]:
import pandas as pd
import numpy as np
import os
def find_lut_o3(o3):
    #根据气体浓度找到目标文件(差值最小)
    lut_folder = '/home/amers/SCIATRAN/SCIATRAN/Execute-4.6.1/DATA_OUT/train_data/O3/'
    files = os.listdir(lut_folder)
    for file in files:
        ex_o3 = file.split('.dat')[0]
        if abs(float(ex_o3) - o3) < 1:  #查找间隔为1
            return os.path.join(lut_folder, file)

def find_lut_h2o(h2o):
    #根据气体浓度找到目标文件(差值最小)
    lut_folder = '/home/amers/SCIATRAN/SCIATRAN/Execute-4.6.1/DATA_OUT/train_data/H2O/'
    files = os.listdir(lut_folder)
    for file in files:
        ex_h2o = file.split('.dat')[0]
        if abs(float(ex_h2o) - h2o) < 0.05:  #查找间隔为0.05
            return os.path.join(lut_folder, file)
        

def load_vod_data(file_path):
    """读取文件并返回波长到VOD的映射，避免重复读取"""
    vod_map = {}
    try:
        with open(file_path, 'r') as f:
            lines = f.readlines()
        # 跳过头部注释（前两行）
        for line in lines[2:]:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    wavelength = float(parts[0])
                    vod = float(parts[1])
                    vod_map[wavelength] = vod
                except ValueError:
                    print(f"Warning: Invalid data in {file_path}: {line.strip()}")
                    continue
    except FileNotFoundError:
        raise FileNotFoundError(f"Cannot find file: {file_path}")
    return vod_map

def find_closest_vod(wavelength: float, vod_map: dict) -> float:
    """从映射中找到与目标波长最接近的VOD"""
    if not vod_map:
        raise ValueError("VOD map is empty")
    closest_wavelength = min(vod_map.keys(), key=lambda x: abs(x - wavelength))
    return vod_map[closest_wavelength]

def extract_vod_by_wavelength(wl_list,o3, h2o):
    """根据波长列表提取O3和H2O的VOD总和"""
    if not wl_list:
        return []


    # 根据气体浓度找到目标文件
    try:
        file_path_o3 = find_lut_o3(o3)
        file_path_h2o = find_lut_h2o(h2o)
    except Exception as e:
        raise RuntimeError(f"Error finding LUT files: {e}")

    # 一次性加载文件数据
    vod_map_o3 = load_vod_data(file_path_o3)
    vod_map_h2o = load_vod_data(file_path_h2o)

    # 计算每个波长的VOD
    vod_list = []
    for wavelength in wl_list:
        try:
            vod_o3 = find_closest_vod(wavelength, vod_map_o3)
            vod_h2o = find_closest_vod(wavelength, vod_map_h2o)
            vod_i = vod_o3 + vod_h2o
            vod_list.append(vod_i)
        except ValueError as e:
            print(f"Warning: Skipping wavelength {wavelength} due to {e}")
            vod_list.append(0.0)  # 

    return vod_list

def generate_angle():
    sin_value = np.random.uniform(0, 1)  
    angle_rad = np.arcsin(sin_value)    
    angle_deg = np.degrees(angle_rad)  
    return round(angle_deg, 2)         

#生成角度list
def generate_angle_list(Nangle, min_val=0, max_val=70):
    half_size = Nangle // 2  # 左侧数据点个数

    # 生成 half_size 个随机数，确保它们递减
    left_half = sorted(np.random.uniform(min_val, max_val, half_size), reverse=True)
    #将left_half中的数据均保留2位小数
    left_half = [round(i, 2) for i in left_half]
    # 如果 Nangle 是奇数，插入 0；否则直接对称翻转
    if Nangle % 2 == 1:
        symmetric_list = left_half + [0] + left_half[::-1]
    else:
        symmetric_list = left_half + left_half[::-1]

    return symmetric_list



# === 生成 SDATA 文件 ===
def generate_sdat(config,sza,phi_forw):
    """ 生成 SDATA 文件 """

    NIP = config["NIP"]
    Nangle = config["Nangle"]
    NWL = config["NWL"]
    wl = config["wl"]

    #判断波长数量是否正确
    if len(wl) != NWL:
        raise ValueError("波长数量不正确")
    #随机生成角度组合
    SZA = round(sza,2)     #generate_angle()                        # 太阳 zenith 角
    phi_forw = round(phi_forw,2)   #round(np.random.uniform(0, 90),2)        # 前向散射相对方位角
    phi_back = 180 - phi_forw
    vza = generate_angle_list(Nangle)             #每个有效观测下的vza组合

    #随机生成高度
    h = float('%.2f' %np.random.uniform(0, 5000)) #高度m

    # 计算测量类型列表
    nip_list = [NIP if i == 1 else 1 for i in config["index"]]  
    temp = [[41,42,43] if i == 1 else [41] for i in config["index"]]
    meas_type_list = [i for k in temp for i in k]  

    nbvm_list = [Nangle] * len(meas_type_list)  
    number = sum(nbvm_list)  
    meas_list = [val for sub in [[0.1] * Nangle + [0.02] * Nangle + [0.02] * Nangle for _ in range(int(sum(nip_list) / NIP))] for val in sub] #数字表示input观测值 I+Q+U or O+U
    phi_list = [val for sub in [[phi_forw] * int(Nangle/2) + [phi_back] * (Nangle - int(Nangle/2)) for _ in range(int(sum(nip_list)))] for val in sub]    #相对方位角
    vza_list = [val for sub in [vza for _ in range(int(sum(nip_list)))] for val in sub]                                                                     # 观测 zenith 角
    


    # 其他固定参数  
    ix, iy, icloudy, icol, irow = "1", "1", "1", "10", "10"
    x, y, MASL, land_percent = "112", "40", str(h), "100"
    
    # 处理字符串数据
    WL = "   ".join(f"{i:.3f}" for i in wl)
    nip = "   ".join(map(str, nip_list))
    meas_type = "   ".join(map(str, meas_type_list))
    nbvm = "   ".join(map(str, np.ones(sum(nip_list), dtype=int) * Nangle))

    sza = "   ".join(map(str, np.ones(NWL, dtype=int) * SZA))
    vza = "   ".join(map(str, vza_list))
    phi = "   ".join(map(str, phi_list))
    meas = "   ".join(map(str, meas_list))

    # 随机生成 O3 (150, 600) DU 和 H2O (0.1, 5) g/cm^2

    o3 = float('%.2f' % np.random.uniform(150, 600))
    h2o = float('%.2f' % np.random.uniform(0.1, 6))
    gaspar = "   ".join(map(str, extract_vod_by_wavelength([i*1000 for i in CONFIG["wl"]],o3, h2o)))  #转为nm单位
    ifcov = "   ".join(map(str, np.zeros(sum(nip_list), dtype=int)))
    cmtrx = "   ".join(map(str, np.ones(number, dtype=int) * 0.01))
    ifmp = "   ".join(map(str, np.zeros(sum(nip_list), dtype=int)))
    mprof = "   ".join(map(str, np.ones(number, dtype=int) * 0.001))

    PIXEL = "   ".join([ix, iy, icloudy, icol, irow, x, y, MASL, land_percent, str(NWL), WL, nip, meas_type, nbvm, sza, vza, phi, meas, gaspar, ifcov, ifmp])

    # 写入 SDATA 文件
    sdat_path = '/media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/' + 'MAP_'+ str(SZA) + '_' + str(phi_forw) + '_' + str(o3)+ '_'+str(h2o) + '_'+ str(h) + '.sdat'
    try:
        with open(sdat_path, "w") as f:
            f.write("SDATA version 2.0\n")
            f.write("  1   1  1  : NX NY NT\n\n")
            f.write("  1   2023-10-04T08:49:28Z  705000.00   0   1   : NPIXELS  TIMESTAMP  HEIGHT_OBS(m)  NSURF  IFGAS    1\n")
            f.write(f"  {PIXEL}")
    except Exception as e:
        raise IOError(f"文件写入失败: {e}")

    print(f"SDATA 文件已保存: {sdat_path}")


# === 设定全局参数===
CONFIG = {
    "NIP": 3,  # 观测类型数量
    "Nangle": 10,  # 一次模拟观测角度数量  (最好不要写奇数)
    "NWL": 6,  # 波长数量
    "wl": [0.443,0.490,0.565,0.670,0.865,1.020],# 个波段 um
    "index": [1, 1, 1, 1, 1, 1],  # 1=偏振波段，0=非偏振
    "bare_soil_file": "/media/amers/2E42853942850735/DataGEO/地表反射率文件/brown fine sandy loam.txt",  
    "grass_file": "/media/amers/2E42853942850735/DataGEO/地表反射率文件/trees deciduous solid decidous.txt",
}       

# === 主程序入口 ===
if __name__ == "__main__":
    for sza in np.arange(56, 60.1, 0.1):
        for phi_forw in np.arange(0, 90, 0.2):
            generate_sdat(CONFIG,sza, phi_forw)  # 生成 SDATA 文件




SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_0.0_462.07_3.35_3055.73.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_0.2_210.55_0.91_2647.47.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_0.4_272.99_0.34_3220.85.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_0.6_403.03_1.06_2218.92.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_0.8_243.11_5.5_1892.91.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_1.0_479.32_2.27_4389.42.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_1.2_592.58_2.16_2176.32.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/MAP_56.0_1.4_224.25_2.96_3168.55.sdat
SDATA 文件已保存: /media/amers/WHX/polder_simulation_r

调用GRASP正向模型

In [10]:
import os
import numpy as np
import yaml
from multiprocessing import Pool
from tqdm import tqdm
import time
import subprocess

TEMP_YAML_DIR = "/media/amers/2E42853942850735/whx/Doctor/NNOE_polder/forward/temp_yaml/"
os.makedirs(TEMP_YAML_DIR, exist_ok=True)  # 确保临时目录存在
GRASP_ROOT = "/media/amers/2E42853942850735/whx/Doctor/NNOE_polder/forward/"

def wyaml(args):
    """处理单个sdat文件并运行GRASP模型"""
    yaml_file = None
    try:
        sdat_file, vc, vc_model, iso, k1, k2, C, ALH = args
        filename = sdat_file.split('.sdat')[0]

        # 读取YAML模板
        with open(os.path.join(GRASP_ROOT, 'settings_forward_models.yml'), 'r') as file:
            result = yaml.safe_load(file)

        # 配置路径参数（使用绝对路径）
        result['input']['file'] = os.path.join('/media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/', sdat_file)
        result['output']['segment']['stream'] = os.path.join('/media/amers/WHX/polder_simulation_results/new_model/', 'forward_results2', f'Radiance_{filename}.txt')

        # 配置模型参数
        result['retrieval']['constraints']['characteristic[1]']['mode[1]']['initial_guess']['value'] = vc_model
        result['retrieval']['constraints']['characteristic[2]']['mode[1]']['initial_guess']['value'] = [vc]
        result['retrieval']['constraints']['characteristic[5]']['mode[1]']['initial_guess']['value'] = [iso] * 6
        result['retrieval']['constraints']['characteristic[5]']['mode[2]']['initial_guess']['value'] = [k1] * 6
        result['retrieval']['constraints']['characteristic[5]']['mode[3]']['initial_guess']['value'] = [k2] * 6
        result['retrieval']['constraints']['characteristic[6]']['mode[1]']['initial_guess']['value'] = [C] * 6
        result['retrieval']['constraints']['characteristic[3]']['mode[1]']['initial_guess']['value'] = ALH

        # 生成临时YAML文件
        yaml_file = os.path.join(TEMP_YAML_DIR, f'loop_forward_{filename}.yml')
        with open(yaml_file, 'w') as f:
            yaml.dump(result, f)

        # 执行GRASP命令并检查返回值
        grasp_cmd = f"cd {GRASP_ROOT} && grasp {yaml_file} > /dev/null 2>&1"
        process = subprocess.run(grasp_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # 检查是否成功
        if process.returncode != 0:
            raise RuntimeError(f"GRASP 运行失败 (exit code {process.returncode})，文件: {sdat_file}")
            
    except Exception as e:
        #print(f"处理 {sdat_file} 时发生错误: {str(e)}")
        with open("error_log.txt", "a") as log_file:
            log_file.write(f"文件 {sdat_file} 运行失败: {str(e)}\n")

    finally:
        # 清理临时YAML文件
        if yaml_file and os.path.exists(yaml_file):
            try:
                os.remove(yaml_file)
            except Exception as e:
                print(f"Error deleting {yaml_file}: {str(e)}")

def generate_task(sdat_file):
    """生成任务参数"""
    # mu, sigma = np.log(0.35), 1
    # # mu, sigma = np.log(0.06), 0.8  
    # vc = float('%.6f' % max(0.0001, min(np.random.lognormal(mu, sigma), 2)))
    vc = float('%.5f' % np.random.uniform(0, 2))
    
    vc_model = np.random.dirichlet(np.array([1, 1, 1, 1]))
    vc_model = [float('%.4f' % x) for x in vc_model]

    ALH = float('%.2f' % np.random.uniform(0.5, 7)) * 1000
    iso = float('%.4f' % np.random.uniform(0.001, 0.8)) 
    # vol = float('%.4f' % np.random.uniform(0.0001, 0.8))
    # geo = float('%.4f' % np.random.uniform(0.0001, 0.8))
    k1 = float('%.4f' % np.random.uniform(0.01, 2.0))
    k2 = float('%.4f' % np.random.uniform(0.01, 2.0))

    C = float('%.3f' % np.random.uniform(1, 8))

    return (sdat_file, vc, vc_model, iso, k1, k2, C, ALH)

if __name__ == "__main__":
    # 配置并行参数
    num_workers = 8 # 根据CPU核心数调整并行度
    print(f"Using {num_workers} workers")

    # 获取任务列表
    sdat_dir = '/media/amers/WHX/polder_simulation_results/new_model/geometry_sdat2/'
    sdat_files = [f for f in os.listdir(sdat_dir) if f.endswith('.sdat')][:]
    
    total_tasks = len(sdat_files)
    start_time = time.time()  # 记录起始时间
    
    with Pool(num_workers) as pool:
        with tqdm(total=total_tasks, desc="Processing", unit="file") as pbar:
            for _ in pool.imap_unordered(wyaml, (generate_task(f) for f in sdat_files)):
                pbar.update(1)  # 仅更新进度条
        
        pool.close()
        pool.join()
    
    print("任务完成，总耗时: {:.2f} 秒".format(time.time() - start_time))

Using 8 workers


Processing: 100%|██████████| 41400/41400 [3:20:16<00:00,  3.45file/s]  

任务完成，总耗时: 12016.92 秒


提取参数构建数据集

In [11]:
import re
import pandas as pd
import os
from tqdm import tqdm
def extract_fitting_data(lines):
    """提取 FITTING 部分数据（sza, vza, fis, sca, I, Q, U）"""
    fitting_data = []
    start_fitting = False
    wavelength = None
    current_block = None

    all_wavelengths = []  # 存储所有波长
    all_fitting_data = {}  # 以波长为键存储I, Q, U数据

    for i, line in enumerate(lines):
        if "*** FITTING from sdata structure ***" in line:
            start_fitting = True
            continue
        if start_fitting:
            if "wavelength #" in line:
                parts = re.split(r'\s+', line.strip())
                wavelength = float(parts[-2])  # 解析波长值
                if wavelength not in all_wavelengths:
                    all_wavelengths.append(wavelength)
                    all_fitting_data[wavelength] = {"I": [], "Q": [], "U": []}
                continue
            if line.strip() == "" or "pixel #" in line:
                continue
            if "#      sza" in line:
                if "meas_I" in line:
                    current_block = "I"
                elif "meas_Q" in line:
                    current_block = "Q"
                elif "meas_U" in line:
                    current_block = "U"
                continue
            if current_block and line.strip():
                parts = re.split(r'\s+', line.strip())
                if len(parts) >= 6:
                    # 解析几何参数 + 观测值
                    sza, vza, fis, sca = map(float, parts[1:5])
                    meas_val = float(parts[6])  # 观测值
                    all_fitting_data[wavelength][current_block].append([sza, vza, fis, sca, meas_val])

    return all_wavelengths, all_fitting_data

def extract_key_value_block(lines, start_keyword, stop_keyword=None):
    """查找从start_keyword开始，到空行或stop_keyword结束的块，并解析成二维list"""
    block = []
    start = None
    for i, line in enumerate(lines):
        if start is None and start_keyword in line:
            start = i + 1
        elif start is not None:
            if stop_keyword and stop_keyword in line:
                break
            if not line.strip():
                break
            block.append(line.strip())
    data = [list(filter(None, re.split(r'\s+', l))) for l in block]
    return data

def extract_single_value(lines, start_keyword):
    """查找start_keyword所在行，下一个非空行返回其第二个数值"""
    for i, line in enumerate(lines):
        if start_keyword in line:
            j = i + 1
            while j < len(lines):
                if lines[j].strip():
                    parts = re.split(r'\s+', lines[j].strip())
                    if len(parts) >= 2:
                        return parts[1]  
                j += 1
    return None

def parse_radiance_file(filename,filepath,output_csv):
    """解析 Radiance_MAP 文件，提取所需信息"""
    with open(filepath, "r") as f:
        lines = f.readlines()

    extracted = {}
    # 根据文件名提取气体浓度
    o3 = filename.split("_")[4]
    extracted["o3"] = o3
    h2o = filename.split("_")[5]
    extracted["h2o"] = h2o
    dem = filename.split("_")[6].split(".txt")[0]
    extracted["dem"] = dem

    # 气溶胶组分比例
    model_frac_block = extract_key_value_block(lines, "Model fraction in total concentration for Particle component 1", "")
    extracted["model_fraction"] = [row[1] for row in model_frac_block[:4]]

    # 气溶胶浓度 & 高度
    extracted["aerosol_volume"] = extract_single_value(lines, "Aerosol volume concentration")
    extracted["aerosol_height"] = extract_single_value(lines, "Aerosol profile mean height")

    # AOD, SSA, AAOD (560nm)
    aod_data = extract_key_value_block(lines, "Wavelength (um), AOD_Total", "Wavelength (um), SSA_Total")
    ssa_data = extract_key_value_block(lines, "Wavelength (um), SSA_Total", "Wavelength (um), AAOD_Total")
    aaod_data = extract_key_value_block(lines, "Wavelength (um), AAOD_Total", "Wavelength (um), REAL Ref. Index")
    aod_fine = extract_key_value_block(lines, "Wavelength (um),  aod_fine", "Wavelength (um),  aod_coarse")
    aod_coarse = extract_key_value_block(lines, "Wavelength (um),  aod_coarse", "Aerosol type:")
    #ref index real & imaginary
    ref_index_real_data = extract_key_value_block(lines, "Wavelength (um), REAL Ref. Index", "Wavelength (um), IMAG Ref. Index")
    ref_index_imag_data = extract_key_value_block(lines, "Wavelength (um), IMAG Ref. Index")

    extracted["AOD_at_565nm"] = [row[1] for row in aod_data if float(row[0]) == 0.565][0]
    extracted["SSA_at_565nm"] = [row[1] for row in ssa_data if float(row[0]) == 0.565][0]
    extracted["AAOD_at_565nm"] = [row[1] for row in aaod_data if float(row[0]) == 0.565][0]
    extracted["AOD_fine"] = [row[1] for row in aod_fine if float(row[0]) == 0.565][0]
    extracted["AOD_coarse"] = [row[1] for row in aod_coarse if float(row[0]) == 0.565][0]
    extracted["mr"] = [row[1] for row in ref_index_real_data if float(row[0]) == 0.565][0]
    extracted["mi"] = [row[1] for row in ref_index_imag_data if float(row[0]) == 0.565][0]

    # 提取 BRDF, BPDF 参数
    brdf_data = extract_key_value_block(lines, "Wavelength (um), BRDF parameters", "BPDF")
    bpdf_data = extract_key_value_block(lines, "Wavelength (um), BPDF parameters")

    extracted["BRDF_params_1"] = brdf_data[1][1] if brdf_data else None
    extracted["BRDF_params_2"] = brdf_data[8][1] if brdf_data else None # 1 +NWL(13)+1 = 15
    extracted["BRDF_params_3"] = brdf_data[15][1] if brdf_data else None
    extracted["BPDF_params"] = bpdf_data[1][1] if bpdf_data else None

    # 提取所有波长的 FITTING 数据
    wavelengths, fitting_data = extract_fitting_data(lines)

    # 组装最终数据
    final_data = []
    for i in range(len(fitting_data[wavelengths[0]]["I"])):
        row = [
            fitting_data[wavelengths[0]]["I"][i][0],  # sza
            fitting_data[wavelengths[0]]["I"][i][1],  # vza
            fitting_data[wavelengths[0]]["I"][i][2],  # fis
            fitting_data[wavelengths[0]]["I"][i][3],  # sca
            extracted["BRDF_params_1"], extracted["BRDF_params_2"], extracted["BRDF_params_3"], extracted["BPDF_params"],
            extracted["aerosol_volume"], *extracted["model_fraction"],extracted["aerosol_height"],
            extracted["AOD_at_565nm"], extracted["SSA_at_565nm"], extracted["AAOD_at_565nm"],extracted['mr'],extracted['mi'],
            extracted['AOD_fine'], extracted['AOD_coarse'],
            extracted['o3'],extracted['h2o'],extracted['dem']
        ]

        # 追加不同波长的 I, Q, U
        for wl in wavelengths:
            row.append(fitting_data[wl]["I"][i][4])  # I
        for wl in wavelengths:
            row.append(fitting_data[wl]["Q"][i][4])  # Q
        for wl in wavelengths:
            row.append(fitting_data[wl]["U"][i][4])  # U

        final_data.append(row)

    # 生成列名
    columns = ["sza", "vza", "fis", "sca", "BRDF1", "BRDF2", "BRDF3", "BPDF",
               "aerosol_volume", "BB", "Urban", "Ocean", "Dust","ALH",
               "AOD_560nm", "SSA_560nm", "AAOD_560nm", "mr", "mi","aod_fine","aod_coarse", "o3", "h2o", "dem"]
    columns += [f"{wl}_I" for wl in wavelengths] + [f"{wl}_Q" for wl in wavelengths] + [f"{wl}_U" for wl in wavelengths]

    # 存入 CSV（追加模式）
    df_output = pd.DataFrame(final_data, columns=columns)
    file_exists = os.path.exists(output_csv)  # 检查文件是否已存在
    df_output.to_csv(output_csv, mode='a', index=False, header=not file_exists)  # 仅首次写入 header


# 批量处理多个 txt 文件
if __name__ == "__main__":
    dir_result = "/media/amers/WHX/polder_simulation_results/new_model/forward_results2/"
    output_csv = "/media/amers/WHX/polder_simulation_results/new_model/train_data_for_polder_new.csv"
    txt_files = [f for f in os.listdir(dir_result) if f.endswith(".txt")]

    for result_file in tqdm(txt_files, desc="Processing files", unit="file"):
        try:
            parse_radiance_file(result_file,os.path.join(dir_result, result_file), output_csv)
        except Exception as e:
            print(f"Error processing {result_file}: {e}")
    print("所有文件处理完成。")





Processing files: 100%|██████████| 41385/41385 [22:06<00:00, 31.20file/s] 

所有文件处理完成。


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

traindata_sample_fracs = {
    "train_data_for_polder.csv":1,
    "train_data_polder_largeVZA.csv": 0.5,

}


traindata_files = [

    '/media/amers/WHX/polder_simulation_results/train_data_for_polder.csv'
]

chunksize = 100_000
wavelengths = ["0.443", "0.49", "0.565", "0.67", "0.865", "1.02"]

def process_chunk(chunk):
    chunk["ALH"] = chunk["ALH"] / 1000
    chunk['dem'] = chunk['dem'] / 1000
    components = ["BB", "Urban", "Ocean", "Dust"]
    for comp in components:
        chunk[f"vc_{comp}"] = chunk["aerosol_volume"] * chunk[comp]

    chunk["k1"] = chunk["BRDF2"]
    chunk["k2"] = chunk["BRDF3"]
    # chunk["sza"]  = chunk["sza"] * np.pi / 180
    # chunk["vza"]  = chunk["vza"] * np.pi / 180
    # chunk["fis"]  = chunk["fis"] * np.pi / 180
    # chunk["sca"]  = chunk["sca"] * np.pi / 180

    for wl in wavelengths:
        I = chunk[f"{wl}_I"]
        Q = chunk[f"{wl}_Q"]
        U = chunk[f"{wl}_U"]
        with np.errstate(divide='ignore', invalid='ignore'):
            dolp = np.sqrt(Q**2 + U**2) / I
            dolp = np.where(I != 0, dolp, np.nan)
            cossza = np.cos(chunk["sza"]* np.pi / 180)
            relectance = I / cossza
        chunk[f"{wl}_DOLP"] = dolp
        chunk[f"{wl}_Reflectance"] = relectance

    cols_to_drop = []
    for wl in wavelengths:
        cols_to_drop.extend([f"{wl}_Q", f"{wl}_U", f"{wl}_I"])
    chunk.drop(columns=cols_to_drop, inplace=True)

    return chunk

df_list = []

# 遍历多个文件
for traindata in traindata_files:

    filename = Path(traindata).name
    frac = traindata_sample_fracs.get(filename, 1)  # 默认用 0.1

    for chunk in pd.read_csv(traindata, chunksize=chunksize):
        sampled = chunk.sample(frac=frac, random_state=88)

        mask = (
            (sampled["AOD_560nm"] <= 5) &
            (sampled["AOD_560nm"].notna())
        )
        sampled = sampled.loc[mask]

        if sampled.empty:
            continue

        processed = process_chunk(sampled)
        df_list.append(processed)

# 合并所有数据
df = pd.concat(df_list, ignore_index=True)

# 修正角度为 0 的情况
# df.loc[df['sza'] == 0, 'sza'] = 0.001
# df.loc[df['vza'] == 0, 'vza'] = 0.001

# 过滤 0 < reflectance < 1 的异常值
df = df[(df['0.443_Reflectance'] > 0) & (df['0.443_Reflectance'] < 1)]
df = df[(df['0.49_Reflectance'] > 0) & (df['0.49_Reflectance'] < 1)]
df = df[(df['0.565_Reflectance'] > 0) & (df['0.565_Reflectance'] < 1)]
df = df[(df['0.67_Reflectance'] > 0) & (df['0.67_Reflectance'] < 1)]
df = df[(df['0.865_Reflectance'] > 0) & (df['0.865_Reflectance'] < 1)]
df = df[(df['1.02_Reflectance'] > 0) & (df['1.02_Reflectance'] < 1)]
# 过滤 DOLP 的异常值
df = df[(df['0.443_DOLP'] > 0) & (df['0.443_DOLP'] < 1)]
df = df[(df['0.49_DOLP'] > 0) & (df['0.49_DOLP'] < 1)]
df = df[(df['0.565_DOLP'] > 0) & (df['0.565_DOLP'] < 1)]
df = df[(df['0.67_DOLP'] > 0) & (df['0.67_DOLP'] < 1)]
df = df[(df['0.865_DOLP'] > 0) & (df['0.865_DOLP'] < 1)]
df = df[(df['1.02_DOLP'] > 0) & (df['1.02_DOLP'] < 1)]


# 保存为单个 parquet 文件
df.to_parquet("/media/amers/WHX/polder_simulation_results/train_data_for_polder.parquet", engine="pyarrow", compression="snappy")
df

,sza,vza,fis,sca,BRDF1,BRDF2,BRDF3,BPDF,aerosol_volume,BB,...,0.49_DOLP,0.49_Reflectance,0.565_DOLP,0.565_Reflectance,0.67_DOLP,0.67_Reflectance,0.865_DOLP,0.865_Reflectance,1.02_DOLP,1.02_Reflectance
0,28.2,12.14,159.6,140.21,0.1866,1.9094,0.1735,0.851,0.11085,0.040396,...,0.065188,0.178497,0.049987,0.146805,0.033778,0.149358,0.020373,0.149381,0.015060,0.147668
1,0.3,37.70,48.7,142.50,0.6322,0.0175,0.9827,5.727,0.38922,0.114900,...,0.095757,0.158272,0.087102,0.133022,0.072309,0.132612,0.053161,0.129452,0.042223,0.126972
2,0.3,16.51,91.4,163.48,0.5411,1.5279,0.7075,4.345,0.07079,0.045305,...,0.006721,0.377675,0.005018,0.376655,0.003867,0.404106,0.002957,0.429166,0.002542,0.437196
3,0.4,12.54,155.1,167.10,0.0481,0.6331,0.5721,1.255,0.23138,0.186300,...,0.013075,0.099591,0.011918,0.077441,0.009798,0.069560,0.005095,0.062988,0.001751,0.059799
4,11.9,10.38,78.0,165.98,0.3611,0.8595,0.2356,4.402,0.51940,0.005300,...,0.004198,0.320958,0.002378,0.308786,0.000715,0.325373,0.001583,0.336788,0.002559,0.337912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6775685,4.4,52.54,51.4,130.12,0.4496,1.4800,0.8489,0.466,0.33742,0.026300,...,0.154540,0.152760,0.140083,0.120936,0.114031,0.111719,0.085543,0.097670,0.067417,0.091122
6775686,33.7,27.20,36.1,160.89,0.2053,0.0544,0.8346,0.755,0.42420,0.223400,...,0.025230,0.152004,0.026556,0.134418,0.025706,0.129442,0.018529,0.128096,0.011771,0.128036
6775687,69.3,31.34,82.1,111.64,0.6995,1.7715,0.9464,8.546,0.44794,0.018300,...,0.249138,0.320447,0.220833,0.259642,0.164690,0.275915,0.112124,0.280642,0.087032,0.277120
6775688,25.0,36.97,85.8,137.96,0.0820,0.2611,0.2625,1.125,0.39540,0.095210,...,0.127738,0.130971,0.117266,0.102497,0.094651,0.099403,0.064952,0.094230,0.047961,0.090741
